# RAG

#### 📊 Part One - Data Preparation

In [ ]:
# %pip install llama-index llama-index-embeddings-cohere llama-index-vector-stores-pinecone pinecone-client pandas llama-index-core

In [ ]:
# ===== NETFREE SSL FIX =====
import ssl
import urllib3
urllib3.disable_warnings()

original_connect = urllib3.connection.HTTPSConnection.connect
def patched_connect(self):
    self.cert_reqs = ssl.CERT_NONE
    self.assert_hostname = False
    original_connect(self)
urllib3.connection.HTTPSConnection.connect = patched_connect
# ===========================

In [ ]:
from datetime import datetime
import os
from dotenv import load_dotenv

from llama_index.core import SimpleDirectoryReader, Settings, VectorStoreIndex
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.llms.cohere import Cohere
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.postprocessor import SimilarityPostprocessor, LongContextReorder
from llama_index.vector_stores.pinecone import PineconeVectorStore
from pinecone import Pinecone
from llama_index.core.llms import ChatMessage

from llama_index.core.workflow import Event, StartEvent, StopEvent, Workflow, step, Context

from llama_index.core.program import LLMTextCompletionProgram
from llama_index.core import Settings
from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from datetime import datetime
import json

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

* 1⃣ Loading



In [28]:

folders = {
    "claude": "../DevTaskManager/.claude",
    "kiro": "../DevTaskManager/.kiro"
}

all_docs = []

for tool_name, path in folders.items():
    if os.path.exists(path):
        reader = SimpleDirectoryReader(input_dir=path, recursive=True,exclude_hidden=False)
        docs = reader.load_data()
        
        for doc in docs:
            file_path = doc.metadata.get("file_path", "")
            
            doc.metadata["coding_tool"] = tool_name
            doc.metadata["file_name"] = os.path.basename(file_path)
            
            if os.path.exists(file_path):
                mtime = os.path.getmtime(file_path)
                doc.metadata["last_modified"] = datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M')   

            doc.excluded_embed_metadata_keys = ["last_modified", "file_path"]
            doc.excluded_llm_metadata_keys = ["file_path"]

        all_docs.extend(docs)
        print(f"✅ Loaded {len(docs)} files from tool: {tool_name}")
    else:
        print(f"⚠️ Warning: The folder {path} was not found.")

print(f"Total documents in the system: {len(all_docs)}")

✅ Loaded 8 files from tool: claude
✅ Loaded 6 files from tool: kiro
Total documents in the system: 14


* 2⃣ Chunking



In [ ]:

splitter = MarkdownNodeParser()

all_nodes = splitter.get_nodes_from_documents(all_docs, show_progress=True)

print(f"Total nodes created: {len(all_nodes)}")

* 3⃣ Embedding



In [ ]:

load_dotenv(r"C:\תכנות שנה ב\דוברות AI\Rag project\RAG\.env")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")

embed_model = CohereEmbedding(
    model_name="embed-multilingual-v3.0",
    api_key=COHERE_API_KEY,
    input_type="search_document" 
)
Settings.embed_model = embed_model

Settings.llm = Cohere(
    model="command-r-plus-08-2024",
    api_key=COHERE_API_KEY,
    max_tokens=500,
)


* 4⃣ VecotrStoreIndex



* 5⃣ Pinecone



In [ ]:

INDEX_NAME = "my-rag-index1"
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=PINECONE_API_KEY)

existing_indexes = [i.name for i in pc.list_indexes()]
print(existing_indexes) 

if INDEX_NAME not in existing_indexes:
    raise ValueError(f"Pinecone index '{INDEX_NAME}' does not exist.")


pinecone_index = pc.Index(INDEX_NAME)

vector_store = PineconeVectorStore(pinecone_index=pinecone_index)

index_stats = pinecone_index.describe_index_stats()
total_vectors = index_stats.get("total_vector_count", 0)

if total_vectors == 0:
    print(f"The index '{INDEX_NAME}' is empty. Uploading {len(all_nodes)} nodes...")
    index = VectorStoreIndex(
        all_nodes,
        vector_store=vector_store,
        show_progress=True
    )
    print("Index created and synced with Pinecone successfully!")
else:
    print(f"The index '{INDEX_NAME}' already contains {total_vectors} vectors.")
    index = VectorStoreIndex.from_vector_store(vector_store)
    print("Index loaded from Pinecone successfully!")

 #### 1⃣ Data Extraction



##### Structured Data Extractor

In [ ]:
# מודל עבור ה-Source הפנימי של כל אייטם
class ItemSource(BaseModel):
    tool: str
    file: str
    anchor: Optional[str] = None
    line_range: List[int] = Field(default_factory=lambda: [0, 0])

# מודל עבור החלטה
class Decision(BaseModel):
    id: str
    title: str
    summary: str
    tags: List[str]
    source: ItemSource
    observed_at: str

# מודל עבור חוק
class Rule(BaseModel):
    id: str
    rule: str
    scope: str
    notes: Optional[str] = None
    source: ItemSource
    observed_at: str

# מודל עבור אזהרה
class WarningItem(BaseModel):
    id: str
    area: str
    message: str
    severity: str
    source: ItemSource
    observed_at: str

# הסכמה שהאקסטרקטור מחפש בכל Node
class NodeExtractionResult(BaseModel):
    decisions: List[Decision] = []
    rules: List[Rule] = []
    warnings: List[WarningItem] = []

In [ ]:
async def extract_to_full_schema(nodes):

    prompt_template_str = """
    You are a Senior System Architect and Code Analyst. 
    Your mission is to extract the "Mental Model" of the project from the provided snippet.
    
    CATEGORIES:
    - Decisions: Architectural choices (e.g., "Uses Express", "JWT for auth", "Bcrypt for passwords").
    - Rules: Patterns and standards (e.g., "Routes are defined in /routes", "Endpoints use POST/GET", "Controllers handle business logic").
    - Warnings: Risks or "don't do" (e.g., "Sensitive .env data", "Missing validation", "Critical database migrations").

    File context: {file_name}
    Snippet: {context_str}

    INSTRUCTIONS:
    1. Be GREEDY: If you see a recurring way of doing things, extract it as a 'Rule' even if it's not labeled as one.
    2. API FOCUS: Specifically list all API endpoints (e.g., /api/login) and their HTTP methods found in the code.
    3. Technical: Focus on naming conventions, folder structures, and tool integrations.
    """

    program = LLMTextCompletionProgram.from_defaults(
        output_cls=NodeExtractionResult,
        prompt_template_str=prompt_template_str,
        llm=Settings.llm
    )

    full_json = {
        "schema_version": "1.0",
        "generated_at": datetime.now().isoformat(),
        "sources": [],
        "items": {"decisions": [], "rules": [], "warnings": []}
    }

    seen_files = set()
    print(f"🚀 Starting extraction on {len(nodes)} nodes...")

    for i, node in enumerate(nodes):
        file_path = node.metadata.get("file_path") or node.metadata.get("file_name", "unknown")
        
        if file_path not in seen_files:
            full_json["sources"].append({
                "tool": node.metadata.get("coding_tool", "unknown"),
                "files": [{
                    "path": file_path,
                    "last_modified": node.metadata.get("last_modified", "n/a")
                }]
            })
            seen_files.add(file_path)

        try:
            output = program(context_str=node.get_content(), file_name=file_path)

            if output.decisions:
                full_json["items"]["decisions"].extend([d.model_dump() for d in output.decisions])
            if output.rules:
                full_json["items"]["rules"].extend([r.model_dump() for r in output.rules])
            if output.warnings:
                full_json["items"]["warnings"].extend([w.model_dump() for w in output.warnings])
            
            if (i+1) % 5 == 0:
                print(f"✅ Processed {i+1}/{len(nodes)} nodes...")

        except Exception as e:
            print(f"⚠️ Error at node {i}: {e}")
            continue

    with open("project_knowledge_base_V2.json", "w", encoding="utf-8") as f:
        json.dump(full_json, f, indent=2, ensure_ascii=False)

    print(f"🏁 Done! Items found: Decisions({len(full_json['items']['decisions'])}), Rules({len(full_json['items']['rules'])})")
    return full_json

In [ ]:
noise_patterns = ["ok", "done", "file saved", "starting...", "installed"]
smart_nodes = [
    node for node in all_nodes 
    if len(node.get_content()) > 150 
    and not any(pattern == node.get_content().strip().lower() for pattern in noise_patterns)
]

print(f"Ready to process {len(smart_nodes)} high-quality nodes.")

##### createing structured file

In [ ]:

if os.path.exists("project_knowledge_base_v2.json"):
    print("✅ Found existing V2 Knowledge Base. Skipping extraction.")
    with open("project_knowledge_base_v2.json", "r", encoding="utf-8") as f:
        structured_data = json.load(f)
else:
    print("🔍 Knowledge Base not found. Starting extraction (this will take a few minutes)...")
    structured_data = await extract_to_full_schema(smart_nodes)

In [ ]:

retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=5,
)
node_postprocessor = SimilarityPostprocessor(similarity_cutoff=0.42)
reorder = LongContextReorder()


##### Events


In [ ]:

class NeedRewriteEvent(Event):
    query: str

class RewriteQueryEvent(Event):
    rewritten_query: str

class RetrievalEvent(Event):
    nodes: list
    attempt: int = 1

class ValidationEvent(Event):
    nodes: list

class NeedMoreContextEvent(Event):
    reason: str
    attempt: int = 1


class StructuredResponseEvent(Event):
    original_question: str

class StructuredQueryEvent(Event):
    category: str  
    filter_keyword: str 
    original_question: str

class ExecutionResultEvent(Event):
    filtered_data: list
    original_question: str
    

class FinalContextEvent(Event):
    """אירוע משותף המכיל את המידע הגולמי שנאסף (מכל מקור שהוא)"""
    context_text: str
    original_question: str
    source_type: str 
   

##### Workflow 

In [ ]:
from llama_index.core.llms import ChatMessage

class AgenticCodingWorkflow(Workflow):
    def __init__(self, retriever, node_postprocessor, reorder, index, structured_json):
        super().__init__()
        self.retriever = retriever
        self.node_postprocessor = node_postprocessor
        self.reorder = reorder
        self.index = index
        self.structured_json = structured_json

# |||||||||||||||||||||||||||||--- Routing---|||||||||||||||||||||||||||||||

    @step
    async def route_query(self, ctx: Context, ev: StartEvent) -> NeedRewriteEvent | StructuredResponseEvent:
        logger.info(f"🚦 Routing query: {ev.query}")
        
        prompt = f"""
        Analyze the question: '{ev.query}'
        Determine if it requires a structured list of technical decisions, rules, or warnings.
        Reply ONLY 'STRUCTURED' if it's a list/rules/decisions request.
        Reply ONLY 'SEARCH' for general explanations or specific technical 'how-to'.
        """
        response = await Settings.llm.achat([ChatMessage(role="user", content=prompt)])
        choice = str(response.message.content).strip().upper()

        if "STRUCTURED" in choice:
            return StructuredResponseEvent(original_question=ev.query)
        else:
            return NeedRewriteEvent(query=ev.query)

#|||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||

#|||||||||||||||||||||||--- מסלול ה-STRUCTURED --||||||||||||||||||||||||||||||

    @step
    async def generate_structured_query(self, ctx: Context, ev: StructuredResponseEvent) -> StructuredQueryEvent:
        logger.info(f"🧠 Generating technical query for: {ev.original_question}")
        
        prompt = f"""
        Based on the question: '{ev.original_question}'
        We have a JSON with 3 categories: 'decisions', 'rules', 'warnings'.
        
        Identify:
        1. The most relevant CATEGORY (decisions, rules, or warnings).
        2. A single KEYWORD to filter the content (e.g., 'RTL', 'DB', 'Auth').
        
        Format your response exactly like this:
        CATEGORY: <category_name>
        KEYWORD: <filter_word>
        """
        
        response = await Settings.llm.achat([ChatMessage(role="user", content=prompt)])
        res_text = str(response.message.content).upper()
        
        category = "decisions" if "DECISION" in res_text else "rules" if "RULE" in res_text else "warnings"
        
        keyword = ""
        if "KEYWORD:" in res_text:
            keyword = res_text.split("KEYWORD:")[-1].strip()
        
        return StructuredQueryEvent(
            category=category, 
            filter_keyword=keyword, 
            original_question=ev.original_question
        )
    
    @step
    async def execute_structured_query(self, ctx: Context, ev: StructuredQueryEvent) -> FinalContextEvent:
        logger.info(f"⚙️ Executing Python filter on category: {ev.category} with keyword: {ev.filter_keyword}")
        
        all_items = self.structured_json.get("items", {}).get(ev.category, [])
        
        filtered = [
            item for item in all_items 
            if ev.filter_keyword.lower() in str(item).lower()
        ]
        
        return FinalContextEvent(
            context_text=json.dumps(filtered[:15], indent=2, ensure_ascii=False),
            original_question=ev.original_question,
            source_type="Structured JSON"
        )
    
#||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||

# ||||||||||||||||||||||||||||||--- מסלול ה-SEARCH ---||||||||||||||||||||||||||||

    @step
    async def rewrite_query(self, ctx: Context, ev: NeedRewriteEvent) -> RewriteQueryEvent|StopEvent:
        logger.info(f"📝 rewrite_query - query received: {ev.query}")

        if not ev.query or not ev.query.strip():
            return StopEvent(result="השאלה לא נכתבה מחדש בהצלחה.")
        
        rewrite_prompt = f"""
        Rewrite the user question to make it better for semantic search.
        Question:
        {ev.query}
        Return ONLY the improved query text, no preamble
        """
        response = await Settings.llm.achat([ChatMessage(role="user", content=rewrite_prompt)])
        rewritten_query = str(response.message.content).strip()

        await ctx.store.set("original_user_query", ev.query)

        logger.info(f"📝 rewrite_query - rewritten query: {rewritten_query}")
        return RewriteQueryEvent(rewritten_query=rewritten_query)


    @step
    async def ingest(self, ctx: Context, ev: RewriteQueryEvent) -> RetrievalEvent | StopEvent:
        logger.info(f"🔍 ingest - retrieving nodes for: {ev.rewritten_query}")

        await ctx.store.set("rewritten_query", ev.rewritten_query)
        await ctx.store.set("attempt_count", 1)

        nodes = self.retriever.retrieve(ev.rewritten_query)
        logger.info(f"✅ ingest - found {len(nodes)} nodes")
        return RetrievalEvent(nodes=nodes, attempt=1)
    

    @step
    async def validate_and_route(self, ctx: Context, ev: RetrievalEvent) -> ValidationEvent | NeedMoreContextEvent:
        logger.info(f"🔎 validate - attempt {ev.attempt}, {len(ev.nodes)} nodes")

        filtered_nodes = self.node_postprocessor.postprocess_nodes(ev.nodes)
        filtered_nodes = self.reorder.postprocess_nodes(filtered_nodes)
        
        logger.info(f"🔎 validate - after postprocessing: {len(filtered_nodes)} nodes")

        await ctx.store.set("last_nodes", filtered_nodes)
        if not filtered_nodes:
            return NeedMoreContextEvent(reason="no_results", attempt=ev.attempt)

        scores = [n.score for n in filtered_nodes if n.score is not None]
        avg_score = sum(scores)/len(scores) if scores else 0
        top_score = max(scores) if scores else 0
        unique_sources = {
            (n.metadata.get("coding_tool"), n.metadata.get("file_name"))
            for n in filtered_nodes
        }

        if avg_score < 0.50 or (top_score < 0.65 and len(unique_sources) < 2):
            logger.warning(f"⚠️ validate - weak context, avg_score={avg_score:.2f}")
            return NeedMoreContextEvent(reason="weak_context", attempt=ev.attempt)
        
        logger.info(f"✅ validate - passed with {len(filtered_nodes)} nodes")
        return ValidationEvent(nodes=filtered_nodes)



    @step
    async def search_retry(self, ctx: Context, ev: NeedMoreContextEvent) -> RetrievalEvent | StopEvent:
        query = await ctx.store.get("rewritten_query")
        if ev.attempt > 2:
            return StopEvent(result="לא נמצא מידע מספיק לאחר מספר ניסיונות.")
        
        broad_retriever = VectorIndexRetriever(index=self.index, similarity_top_k=10)
        nodes = broad_retriever.retrieve(query)
        return RetrievalEvent(nodes=nodes, attempt=ev.attempt + 1)



    @step
    async def synthesize(self, ctx: Context, ev: ValidationEvent) -> FinalContextEvent:
        query = await ctx.store.get("original_user_query")

        top_nodes = sorted(ev.nodes, key=lambda n: n.score if n.score else 0, reverse=True)[:4]
        context_list = [f"[{n.metadata.get('coding_tool','Unknown')} | {n.metadata.get('file_name','File')}] "
                        f"(modified at {n.metadata.get('last_modified','N/A')}): {n.get_content()}" 
                        for n in top_nodes]
        full_context = "\n\n".join(context_list)

        return FinalContextEvent(
            context_text=full_context,
            original_question=query,
            source_type="Vector Search"
        )
    
    #|||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||

    #|||||||||||||||||||||||||||||||||||| final answer step ||||||||||||||||||||||||||||||||


    @step
    async def generate_final_response(self, ctx: Context, ev: FinalContextEvent) -> StopEvent:
        logger.info(f"✍️ Generating final answer from source: {ev.source_type}")
        
        if not ev.context_text or ev.context_text == "[]":
            return StopEvent(result="מצטער, חיפשתי בתיעוד הפרויקט (סמנטית ומובנית) ולא מצאתי מידע רלוונטי.")

        chat_history = await ctx.store.get("chat_history", default=[])
        prompt = f"""
        You are a senior development assistant who analyzes project documentation.
        Previous conversation history:
        {chat_history}

        Current question:
        {ev.original_question}

        Source Type used: {ev.source_type}
        
        Information found:
        {ev.context_text}

        Instructions:
        1. Provide a concise, professional, and technical answer.
        2. If there is a conflict between sources, give preference to the most recent information by date.
        3. ALWAYS indicate which tool/file the answer was taken from.
        4. If the source data is from a Structured JSON, present it as a clean, organized list.
        5. If there is no unambiguous answer in the text, say so honestly.
        """
        response = await Settings.llm.achat([ChatMessage(role="user", content=prompt)])
        answer = str(response.message.content)

        chat_history.append({"user": ev.original_question, "assistant": answer})
        await ctx.store.set("chat_history", chat_history[-3:])

        logger.info(f"✅ Final response generated from {ev.source_type}")
        return StopEvent(result=answer)

##### Instantiate Workflow

In [ ]:

wf = AgenticCodingWorkflow(
    retriever=retriever,
    node_postprocessor=node_postprocessor,
    reorder=reorder,
    index=index,
    structured_json=structured_data,
)

##### Gradio interface

In [ ]:

import gradio as gr

async def chat_function(message, history):
    result = await wf.run(query=message)
    return str(result)

demo = gr.ChatInterface(
    fn=chat_function,
    title="Coding Assistant - RAG System",
    description="Ask questions about your coding tools and files!",
    examples=[
        "What does the function in file X do?", 
        "Which programming tools are in the repository?",
        "When was file Y last modified?"
    ]
)

if __name__ == "__main__":
    demo.launch()

2026-03-12 22:24:45,914 - INFO - 🚦 Routing query: Which programming tools are in the repository?
2026-03-12 22:24:46,264 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 429 Too Many Requests"
2026-03-12 22:24:47,563 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 429 Too Many Requests"
2026-03-12 22:24:49,742 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 429 Too Many Requests"
Traceback (most recent call last):
  File "C:\Users\user1\AppData\Roaming\Python\Python312\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user1\AppData\Roaming\Python\Python312\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user1\AppData\Roaming\Python\Python312\site-packages\gradio\